In [1]:
pip install opencv-python easyocr numpy

  Using cached scipy-1.17.1-cp313-cp313-macosx_14_0_arm64.whl.metadata (62 kB)
  Using cached pyyaml-6.0.3-cp313-cp313-macosx_11_0_arm64.whl.metadata (2.4 kB)
  Using cached ninja-1.13.0-py3-none-macosx_10_9_universal2.whl.metadata (5.1 kB)
  Using cached filelock-3.29.0-py3-none-any.whl.metadata (2.0 kB)
  Using cached setuptools-81.0.0-py3-none-any.whl.metadata (6.6 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
  Using cached markupsafe-3.0.3-cp313-cp313-macosx_11_0_arm64.whl.metadata (2.7 kB)
  Using cached lazy_loader-0.5-py3-none-any.whl.metadata (5.9 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.2/46.2 MB 7.9 MB/s  0:00:05 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 8.8 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.

In [2]:
import cv2
import easyocr
import numpy as np
import re
from collections import defaultdict

In [3]:
reader = easyocr.Reader(['en'], gpu=False)

Using CPU. Note: This module is much faster with a GPU.


In [4]:
def extract_number(text):
    text = text.replace(",", "").replace("%", "").replace("m", "").strip()

    # OCR sometimes reads zero as letter O
    text = text.replace("O", "0").replace("o", "0")

    matches = re.findall(r"-?\d+\.?\d*", text)

    if not matches:
        return None

    return float(matches[0])

In [5]:
def box_geometry(box):
    xs = [p[0] for p in box]
    ys = [p[1] for p in box]

    return {
        "left": min(xs),
        "right": max(xs),
        "top": min(ys),
        "bottom": max(ys),
        "cx": sum(xs) / 4,
        "cy": sum(ys) / 4
    }

In [6]:
def preprocess_for_ocr(image):
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    gray = cv2.resize(gray, None, fx=2, fy=2, interpolation=cv2.INTER_CUBIC)
    
    clahe = cv2.createCLAHE(clipLimit=2.5, tileGridSize=(8, 8))
    gray = clahe.apply(gray)
    
    return gray

In [7]:
def ocr_numeric_boxes(image):
    processed = preprocess_for_ocr(image)
    results = reader.readtext(processed)

    numeric_boxes = []

    for box, text, confidence in results:
        if confidence < 0.2:
            continue

        number = extract_number(text)
        if number is None:
            continue

        geo = box_geometry(box)

        numeric_boxes.append({
            "value": number,
            "text": text,
            "confidence": confidence,
            "left": geo["left"],
            "right": geo["right"],
            "top": geo["top"],
            "bottom": geo["bottom"],
            "cx": geo["cx"],
            "cy": geo["cy"],
            "box": box
        })

    return numeric_boxes, processed.shape[1], processed.shape[0]

In [8]:
# def remove_same_y_groups(numbers, image_width, image_height):
#     """
#     Removes numeric labels that look like stacked-bar/data labels.

#     Stacked/data labels often:
#     - share similar y-position
#     - spread across x-position

#     Y-axis labels usually:
#     - share similar x-position
#     - spread across y-position
#     """

#     if len(numbers) < 3:
#         return numbers

#     y_groups = defaultdict(list)
#     y_bin_size = image_height * 0.06

#     for item in numbers:
#         y_bin = int(item["cy"] // y_bin_size)
#         y_groups[y_bin].append(item)

#     labels_to_remove = set()

#     for group in y_groups.values():
#         if len(group) < 3:
#             continue

#         x_positions = [g["cx"] for g in group]
#         y_positions = [g["cy"] for g in group]

#         x_spread = max(x_positions) - min(x_positions)
#         y_spread = max(y_positions) - min(y_positions)

#         # Same y-level, spread horizontally = likely bar/data labels
#         if y_spread < image_height * 0.06 and x_spread > image_width * 0.25:
#             for g in group:
#                 labels_to_remove.add(id(g))

#     return [item for item in numbers if id(item) not in labels_to_remove]


In [9]:
def is_monotonic(values):
    if len(values) < 2:
        return False

    increasing = all(values[i] <= values[i + 1] for i in range(len(values) - 1))
    decreasing = all(values[i] >= values[i + 1] for i in range(len(values) - 1))

    return increasing or decreasing

In [10]:
def is_evenly_spaced(values, tolerance_ratio=0.35):
    if len(values) < 3:
        return False

    values = sorted(values)
    diffs = [values[i + 1] - values[i] for i in range(len(values) - 1)]

    if any(d == 0 for d in diffs):
        return False

    avg_diff = sum(diffs) / len(diffs)

    return all(abs(d - avg_diff) <= avg_diff * tolerance_ratio for d in diffs)

In [11]:
def find_vertical_value_axis(numbers, image_width, image_height):
    """
    Find y-axis labels for vertical bar charts using scoring.

    A good y-axis candidate should:
    - be near the left side
    - be vertically aligned
    - cover enough height
    - have monotonic values
    - have evenly spaced values
    """

    if len(numbers) < 2:
        return []

    groups = defaultdict(list)

    # Group numbers by similar x-position
    bin_size = image_width * 0.04

    for item in numbers:
        x_bin = int(item["right"] // bin_size)
        groups[x_bin].append(item)

    best_group = []
    best_score = 0

    for group in groups.values():
        if len(group) < 2:
            continue

        group = sorted(group, key=lambda x: x["cy"])

        x_positions = [g["right"] for g in group]
        y_positions = [g["cy"] for g in group]
        values = [g["value"] for g in group]

        mean_x = sum(x_positions) / len(x_positions)
        x_spread = max(x_positions) - min(x_positions)
        y_spread = max(y_positions) - min(y_positions)

        score = 0

        # 1. Axis labels are usually near the left side
        if mean_x < image_width * 0.25:
            score += 3

        # 2. Axis labels should have similar x-position
        if x_spread < image_width * 0.12:
            score += 3

        # 3. Axis labels should be spread vertically
        if y_spread > image_height * 0.25:
            score += 3

        # 4. Axis should have at least 3 tick values
        if len(group) >= 3:
            score += 2

        # 5. Axis values usually increase/decrease consistently
        if is_monotonic(values):
            score += 4

        # 6. Axis values often have equal spacing
        if is_evenly_spaced(values):
            score += 4

        # Require minimum score to avoid false positives
        if score > best_score and score >= 8:
            best_score = score
            best_group = group

    return best_group

In [12]:
def find_horizontal_value_axis(numbers, image_width, image_height):
    """
    For horizontal bar charts:
    x-axis labels form a horizontal stack:
    - similar y-position
    - multiple different x-positions
    """

    if len(numbers) < 2:
        return []

    groups = defaultdict(list)
    bin_size = image_height * 0.08

    for item in numbers:
        y_bin = int(item["cy"] // bin_size)
        groups[y_bin].append(item)

    best_group = []

    for group in groups.values():
        if len(group) < 2:
            continue

        group = sorted(group, key=lambda x: x["cx"])

        x_positions = [g["cx"] for g in group]
        y_positions = [g["cy"] for g in group]

        x_spread = max(x_positions) - min(x_positions)
        y_spread = max(y_positions) - min(y_positions)

        if y_spread < image_height * 0.08 and x_spread > image_width * 0.25:
            if not best_group or max(y_positions) > max(g["cy"] for g in best_group):
                best_group = group

    return best_group

In [13]:
def ocr_bottom_left_for_zero(image, image_w, image_h):
    h, w = image.shape[:2]
    
    # Widen slightly to give EasyOCR more character separation context
    crop = image[int(h * 0.70):, :int(w * 0.25)]
    
    if crop.size == 0:
        return False

    gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
    resized = cv2.resize(gray, None, fx=3, fy=3, interpolation=cv2.INTER_CUBIC)
    
    strategies = []
    
    # Strategy 1: raw grayscale — let EasyOCR handle contrast itself
    strategies.append(resized)
    
    # Strategy 2: CLAHE only, no threshold
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(4, 4))
    strategies.append(clahe.apply(resized.copy()))
    
    # Strategy 3: Otsu threshold (background-aware)
    is_dark = np.mean(resized) < 128
    tt = cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU if is_dark else cv2.THRESH_BINARY + cv2.THRESH_OTSU
    _, s3 = cv2.threshold(resized.copy(), 0, 255, tt)
    strategies.append(s3)

    for img in strategies:
        results = reader.readtext(img, allowlist='0123456789.-')  # restrict alphabet
        for _, text, conf in results:
            num = extract_number(text)
            if conf > 0.05 and num is not None and abs(num) < 1.0:
                return True
    return False

In [14]:
def infer_zero_on_axis(axis_values, tolerance=0.01):
    if len(axis_values) < 2:
        return False
    vals = sorted(axis_values)
    step = (vals[-1] - vals[0]) / (len(vals) - 1)
    if step <= 0:
        return False
    extrapolated = vals[0] - step
    return abs(extrapolated) <= abs(step) * tolerance

In [15]:
def hunt_zero_in_full_image(image, image_width, image_height):
    """
    Dedicated pass to find a standalone '0' axis label.
    Restricts OCR alphabet and searches only the left strip.
    """
    h, w = image.shape[:2]
    left_strip = image[:, :int(w * 0.25)]  # only left 25%
    
    gray = cv2.cvtColor(left_strip, cv2.COLOR_BGR2GRAY)
    gray = cv2.resize(gray, None, fx=2, fy=2, interpolation=cv2.INTER_CUBIC)
    clahe = cv2.createCLAHE(clipLimit=2.5, tileGridSize=(8, 8))
    gray = clahe.apply(gray)
    
    results = reader.readtext(gray, allowlist='0123456789.m%')
    for _, text, conf in results:
        num = extract_number(text)
        if conf > 0.05 and num is not None and abs(num) < 1.0:
            return True
    return False

In [16]:
def check_starts_at_zero(image_path, orientation):
    image = cv2.imread(image_path)

    if image is None:
        raise ValueError("Image not found")

    numbers, w, h = ocr_numeric_boxes(image)

    if orientation == "vertical":
        axis_group = find_vertical_value_axis(numbers, w, h)
        axis_type = "y-axis"

    elif orientation == "horizontal":
        axis_group = find_horizontal_value_axis(numbers, w, h)
        axis_type = "x-axis"

    else:
        return {
            "status": "unknown",
            "starts_at_zero": None,
            "reason": "Invalid or unknown orientation"
        }

    if not axis_group:
        return {
            "status": "compliant",
            "starts_at_zero": True,
            "orientation": orientation,
            "axis_type": axis_type,
            "detected_axis_values": [],
            "reason": "No value-axis labels detected; treated as compliant by client rule"
        }

    axis_values = [item["value"] for item in axis_group]

    contains_zero = any(abs(v) < 1.0 for v in axis_values)
    zero_source = "axis_group" if contains_zero else None

    if not contains_zero and orientation == "vertical":
        contains_zero = ocr_bottom_left_for_zero(image, w, h)
        if contains_zero:
            zero_source = "rescue_crop"
            axis_values = axis_values + [0.0]  # now visible in output

    if not contains_zero and orientation == "vertical":
        contains_zero = hunt_zero_in_full_image(image, w, h)
        if contains_zero:
            zero_source = "rescue_hunt"
            axis_values = axis_values + [0.0]  # now visible in output

    reason_map = {
        "axis_group":   "Zero found directly in axis label group",
        "rescue_crop":  "Zero found via bottom-left crop rescue",
        "rescue_hunt":  "Zero found via left-strip digit scan rescue",
        None:           "Value-axis labels detected but zero was not found"
    }

    return {
        "status": "compliant" if contains_zero else "non_compliant",
        "starts_at_zero": contains_zero,
        "orientation": orientation,
        "axis_type": axis_type,
        "detected_axis_values": sorted(axis_values),  # sorted so 0 shows at front
        "zero_source": zero_source,
        "reason": reason_map[zero_source]
    }

In [21]:
result = check_starts_at_zero(image_path="/Users/nguyenanhvu/Documents/AMD-Semester3/GroupProject/IBCS-Scaling/Dataset/Screenshot 2026-05-26 at 10.04.04.png", orientation="horizontal")
print(result)

{'status': 'compliant', 'starts_at_zero': True, 'orientation': 'horizontal', 'axis_type': 'x-axis', 'detected_axis_values': [0.0, 1.0, 219.0, 1889.0], 'zero_source': 'axis_group', 'reason': 'Zero found directly in axis label group'}


In [ ]:
def debug_ocr(image_path):
    image = cv2.imread(image_path)
    h_orig, w_orig = image.shape[:2]
    
    # Show what the preprocessed image looks like
    processed = preprocess_for_ocr(image)
    cv2.imwrite("debug_preprocessed.png", processed)
    
    # Show ALL raw EasyOCR detections, no filtering
    results = reader.readtext(processed)
    print(f"Image original size: {w_orig}x{h_orig}")
    print(f"Preprocessed size: {processed.shape[1]}x{processed.shape[0]}")
    print(f"\nALL detections (including low confidence):")
    for box, text, conf in results:
        geo = box_geometry(box)
        print(f"  text='{text}' conf={conf:.2f} right={geo['right']:.0f} cy={geo['cy']:.0f}")

debug_ocr("/Users/nguyenanhvu/Documents/AMD-Semester3/GroupProject/IBCS-Scaling/Dataset/Screenshot 2026-05-21 at 15.41.56.png")

Image original size: 616x776
Preprocessed size: 1232x1552

ALL detections (including low confidence):
  text='-2.1' conf=0.89 right=1093 cy=27
  text='+31.7' conf=1.00 right=691 cy=121
  text='+4.7' conf=1.00 right=905 cy=121
  text='300m' conf=0.98 right=220 cy=244
  text='289.3' conf=0.66 right=1056 cy=232
  text='+24.9' conf=0.78 right=465 cy=337
  text='-5.2' conf=0.97 right=1121 cy=384
  text='250m' conf=1.00 right=220 cy=429
  text='200m' conf=0.79 right=220 cy=613
  text='3' conf=0.96 right=68 cy=748
  text='322.8' conf=1.00 right=666 cy=753
  text='323.1' conf=0.73 right=886 cy=754
  text='150m' conf=0.95 right=220 cy=798
  text='264.4' conf=0.58 right=441 cy=863
  text='247.1' conf=0.88 right=1112 cy=894
  text='100m' conf=0.91 right=220 cy=982
  text='50m' conf=0.75 right=220 cy=1168
  text='Q1' conf=1.00 right=402 cy=1396
  text='Q2' conf=1.00 right=632 cy=1396
  text='Q3' conf=0.94 right=854 cy=1396
  text='Q4' conf=0.96 right=1080 cy=1396
  text='Quarter' conf=1.00 right=8

In [181]:
def debug_rescue(image_path):
    image = cv2.imread(image_path)
    h, w = image.shape[:2]
    crop = image[int(h * 0.75):, :int(w * 0.20)]
    cv2.imwrite("debug_crop.png", crop)
    
    gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
    gray = cv2.resize(gray, None, fx=3, fy=3, interpolation=cv2.INTER_CUBIC)
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(4, 4))
    gray = clahe.apply(gray)
    is_dark_bg = np.mean(gray) < 128
    thresh_type = cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU if is_dark_bg else cv2.THRESH_BINARY + cv2.THRESH_OTSU
    _, gray_thresh = cv2.threshold(gray, 0, 255, thresh_type)
    cv2.imwrite("debug_crop_thresh.png", gray_thresh)
    
    results = reader.readtext(gray_thresh)
    print(f"Crop size: {crop.shape}, dark_bg={is_dark_bg}")
    print("Rescue OCR results:")
    for _, text, conf in results:
        print(f"  text='{text}' conf={conf:.2f} -> number={extract_number(text)}")

debug_rescue("/Users/nguyenanhvu/Documents/AMD-Semester3/GroupProject/IBCS-Scaling/Dataset/Screenshot 2026-05-21 at 15.41.56.png")

Crop size: (194, 123, 3), dark_bg=False
Rescue OCR results:
  text='dum' conf=0.30 -> number=None
